# 🥈 Silver CRM & ERP — Data Transformation

## Goal
Clean, standardize and validate the CRM and ERP data from the Bronze layer and load the transformed data into Silver Delta Tables.

## Flow
Bronze Delta Tables → Cleaning → Data Quality → Silver Delta Tables

## 1. CRM Customer Information

Read the CRM customer data from Bronze and inspect the data before transformation.

In [0]:
crm_cust_df = spark.table("bronze.crm_cust_info")

display(crm_cust_df)

### Transform CRM Customer Data

In [0]:
from pyspark.sql.functions import col, trim, upper, when, row_number
from pyspark.sql.window import Window

window = Window.partitionBy("cst_id") \
    .orderBy(col("cst_create_date").desc())

crm_cust_df = crm_cust_df \
    .filter(col("cst_id").isNotNull()) \
    .withColumn("flag_last", row_number().over(window)) \
    .filter(col("flag_last") == 1) \
    .drop("flag_last") \
    .withColumn("cst_firstname", trim(col("cst_firstname"))) \
    .withColumn("cst_lastname", trim(col("cst_lastname"))) \
    .withColumn(
        "cst_marital_status",
        when(upper(trim(col("cst_marital_status"))) == "S", "Single")
        .when(upper(trim(col("cst_marital_status"))) == "M", "Married")
        .otherwise("n/a")
    ) \
    .withColumn(
        "cst_gndr",
        when(upper(trim(col("cst_gndr"))) == "F", "Female")
        .when(upper(trim(col("cst_gndr"))) == "M", "Male")
        .otherwise("n/a")
    )

display(crm_cust_df)

### Load to Silver Delta Table

In [0]:
crm_cust_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.crm_cust_info")

In [0]:
%sql
SELECT *
FROM silver.crm_cust_info;

## 2. CRM Product Information

Transform CRM product data from Bronze and load the cleaned data into the Silver Delta Table.

In [0]:
from pyspark.sql.functions import col, substring, regexp_replace, coalesce, lit, trim, upper, when, lead, date_sub
from pyspark.sql.window import Window

crm_prd_df = spark.table("bronze.crm_prd_info")

product_window = Window.partitionBy("prd_key") \
    .orderBy("prd_start_dt")

crm_prd_df = crm_prd_df \
    .withColumn(
        "cat_id",
        regexp_replace(substring(col("prd_key"), 1, 5), "-", "_")
    ) \
    .withColumn(
        "prd_key",
        substring(col("prd_key"), 7, 100)
    ) \
    .withColumn(
        "prd_cost",
        coalesce(col("prd_cost"), lit(0))
    ) \
    .withColumn(
        "prd_line",
        when(upper(trim(col("prd_line"))) == "M", "Mountain")
        .when(upper(trim(col("prd_line"))) == "R", "Road")
        .when(upper(trim(col("prd_line"))) == "S", "Other Sales")
        .when(upper(trim(col("prd_line"))) == "T", "Touring")
        .otherwise("n/a")
    ) \
    .withColumn(
        "prd_end_dt",
        date_sub(
            lead("prd_start_dt").over(product_window),
            1
        )
    )

display(crm_prd_df)

In [0]:
crm_prd_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.crm_prd_info")

In [0]:
%sql
SELECT *
FROM silver.crm_prd_info;

## 3. CRM Sales Details

Transform CRM sales data from Bronze and load the cleaned data into the Silver Delta Table.

In [0]:
from pyspark.sql.functions import col, when, to_date, abs

crm_sales_df = spark.table("bronze.crm_sales_details")

crm_sales_df = crm_sales_df \
    .withColumn(
        "sls_order_dt",
        when(
            (col("sls_order_dt") == "0") |
            (col("sls_order_dt").cast("string").rlike(r"^\d{8}$") == False),
            None
        ).otherwise(
            to_date(col("sls_order_dt").cast("string"), "yyyyMMdd")
        )
    ) \
    .withColumn(
        "sls_ship_dt",
        when(
            (col("sls_ship_dt") == "0") |
            (col("sls_ship_dt").cast("string").rlike(r"^\d{8}$") == False),
            None
        ).otherwise(
            to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd")
        )
    ) \
    .withColumn(
        "sls_due_dt",
        when(
            (col("sls_due_dt") == "0") |
            (col("sls_due_dt").cast("string").rlike(r"^\d{8}$") == False),
            None
        ).otherwise(
            to_date(col("sls_due_dt").cast("string"), "yyyyMMdd")
        )
    ) \
    .withColumn(
        "sls_sales",
        when(
            col("sls_sales").isNull() |
            (col("sls_sales") <= 0) |
            (col("sls_sales") != col("sls_quantity") * abs(col("sls_price"))),
            col("sls_quantity") * abs(col("sls_price"))
        ).otherwise(col("sls_sales"))
    ) \
    .withColumn(
        "sls_price",
        when(
            col("sls_price").isNull() |
            (col("sls_price") <= 0),
            col("sls_sales") / col("sls_quantity")
        ).otherwise(col("sls_price"))
    )

display(crm_sales_df)

In [0]:
crm_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.crm_sales_details")

In [0]:
%sql
SELECT *
FROM silver.crm_sales_details;

## 4. ERP Customer Information

Transform ERP customer data from Bronze and load the cleaned data into the Silver Delta Table.

In [0]:
from pyspark.sql.functions import col, when, upper, trim, current_date, regexp_replace

erp_cust_df = spark.table("bronze.erp_cust_az12")

erp_cust_df = erp_cust_df \
    .withColumn(
        "CID",
        when(
            col("CID").startswith("NAS"),
            regexp_replace(col("CID"), "^NAS", "")
        ).otherwise(col("CID"))
    ) \
    .withColumn(
        "BDATE",
        when(
            col("BDATE") > current_date(),
            None
        ).otherwise(col("BDATE"))
    ) \
    .withColumn(
        "GEN",
        when(upper(trim(col("GEN"))).isin("F", "FEMALE"), "Female")
        .when(upper(trim(col("GEN"))).isin("M", "MALE"), "Male")
        .otherwise("n/a")
    )

display(erp_cust_df)

In [0]:
erp_cust_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.erp_cust_az12")

In [0]:
%sql
SELECT *
FROM silver.erp_cust_az12;

## 5. ERP Customer Location

Transform ERP customer location data from Bronze and load the cleaned data into the Silver Delta Table.

In [0]:
from pyspark.sql.functions import col, trim, when, regexp_replace

erp_loc_df = spark.table("bronze.erp_loc_a101")

erp_loc_df = erp_loc_df \
    .withColumn(
        "CID",
        regexp_replace(col("CID"), "-", "")
    ) \
    .withColumn(
        "CNTRY",
        when(trim(col("CNTRY")) == "DE", "Germany")
        .when(trim(col("CNTRY")).isin("US", "USA"), "United States")
        .when(
            col("CNTRY").isNull() | (trim(col("CNTRY")) == ""),
            "n/a"
        )
        .otherwise(trim(col("CNTRY")))
    )

display(erp_loc_df)

In [0]:
erp_loc_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.erp_loc_a101")

In [0]:
%sql
SELECT *
FROM silver.erp_loc_a101;

## 6. ERP Product Categories

Load ERP product category data from Bronze into the Silver Delta Table.

In [0]:
erp_cat_df = spark.table("bronze.erp_px_cat_g1v2")

display(erp_cat_df)

In [0]:
erp_cat_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.erp_px_cat_g1v2")

In [0]:
%sql
SELECT *
FROM silver.erp_px_cat_g1v2;